In [1]:
import os, random
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from datasets import VOCDataset, make_loaders


In [ ]:
S          = 7
B          = 2
C          = 20
IMG_SIZE   = 448
BATCH_SIZE = 32          
LR         = 2e-5
EPOCHS     = 30

NUM_WORKERS = os.cpu_count()   # 10 on M5
torch.set_num_threads(NUM_WORKERS)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

VOC_TRAIN = [
    ('data/VOC/VOC2007', 'train'),
    ('data/VOC/VOC2012', 'trainval'),
]
VOC_VAL = [('data/VOC/VOC2007', 'test')]

CLASS_NAMES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLS2IDX = {n: i for i, n in enumerate(CLASS_NAMES)}

print(f'device={DEVICE}  workers={NUM_WORKERS}  S={S}  B={B}  C={C}')


device=mps  workers=10  S=7  B=2  C=20


In [3]:
train_ds = VOCDataset(VOC_TRAIN, CLS2IDX, IMG_SIZE, augment_data=True,  cache=True)
val_ds   = VOCDataset(VOC_VAL,   CLS2IDX, IMG_SIZE, augment_data=False, cache=True)
print(f'train={len(train_ds)}  val={len(val_ds)}')

train_dl, val_dl = make_loaders(train_ds, val_ds, BATCH_SIZE, NUM_WORKERS)


  caching 14041 VOC images into RAM...
  done
  caching 4952 VOC images into RAM...
  done
train=14041  val=4952


In [4]:
def cb(ic, oc, k, s=1, p=0):
    return nn.Sequential(
        nn.Conv2d(ic, oc, k, s, p, bias=False),
        nn.BatchNorm2d(oc),
        nn.LeakyReLU(0.1, inplace=True),
    )


class YOLOv1(nn.Module):
    """
    YOLOv1 with GoogLeNet backbone (pretrained on ImageNet).
    The paper states the architecture is "inspired by GoogLeNet".
    GoogLeNet inception5b → 14×14×1024 for 448×448 input,
    which matches the paper's backbone output exactly.
    Detection layers 21-24 + 2 FC are trained from scratch.
    """
    def __init__(self, S=7, B=2, C=20):
        super().__init__()
        self.S = S; self.B = B; self.C = C

        gnet = torchvision.models.googlenet(weights='IMAGENET1K_V1')

        # backbone = GoogLeNet up through inception5b → 14×14×1024 for 448×448 input
        self.backbone = nn.Sequential(
            gnet.conv1,      # 448 → 224
            gnet.maxpool1,   # 224 → 112
            gnet.conv2,
            gnet.conv3,
            gnet.maxpool2,   # 112 → 56
            gnet.inception3a,
            gnet.inception3b,
            gnet.maxpool3,   # 56 → 28
            gnet.inception4a,
            gnet.inception4b,
            gnet.inception4c,
            gnet.inception4d,
            gnet.inception4e,
            gnet.maxpool4,   # 28 → 14
            gnet.inception5a,
            gnet.inception5b,  # → 14×14×1024
        )

        # detection head: paper layers 21-24 (14×14 → 7×7)
        self.head = nn.Sequential(
            cb(1024, 1024, 3, p=1),        # 21
            cb(1024, 1024, 3, s=2, p=1),   # 22  14→7
            cb(1024, 1024, 3, p=1),        # 23
            cb(1024, 1024, 3, p=1),        # 24
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 4096),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, S * S * (B * 5 + C)),
        )

    def forward(self, x):
        return self.classifier(self.head(self.backbone(x))).reshape(-1, self.S, self.S, self.B * 5 + self.C)


def freeze_backbone(model, freeze=True):
    for p in model.backbone.parameters():
        p.requires_grad_(not freeze)
    print(f'backbone {"frozen" if freeze else "unfrozen"}')


_m = YOLOv1(S, B, C)
_o = _m(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE))
print(f'output: {tuple(_o.shape)}  expected (1, {S}, {S}, {B*5+C})')
print(f'params: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M  '
      f'backbone: {sum(p.numel() for p in _m.backbone.parameters())/1e6:.1f}M')
del _m, _o


Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /Users/vineet/.cache/torch/hub/checkpoints/googlenet-1378be20.pth


100%|██████████| 49.7M/49.7M [00:05<00:00, 9.18MB/s]


output: (1, 7, 7, 30)  expected (1, 7, 7, 30)
params: 254.9M  backbone: 5.6M


In [5]:
class YOLOv1Loss(nn.Module):
    def __init__(self, S=7, B=2, C=80):
        super().__init__()
        self.S = S; self.B = B; self.C = C
        self.mse = nn.MSELoss(reduction='sum')

    def forward(self, pred, targets):
        N, S, B, C = pred.shape[0], self.S, self.B, self.C
        device = pred.device

        boxes = pred[..., :B*5].reshape(N, S, S, B, 5)  # (N,S,S,B,5)
        cls_p = pred[..., B*5:]                          # (N,S,S,C)

        gy, gx = torch.meshgrid(
            torch.arange(S, device=device, dtype=torch.float32),
            torch.arange(S, device=device, dtype=torch.float32),
            indexing='ij')

        # decode predicted cx,cy,w,h to absolute (0-1) coords for IoU
        pcx = (torch.sigmoid(boxes[..., 1]) + gx[None,:,:,None]) / S
        pcy = (torch.sigmoid(boxes[..., 2]) + gy[None,:,:,None]) / S
        pw  = torch.sigmoid(boxes[..., 3])
        ph  = torch.sigmoid(boxes[..., 4])
        pc  = boxes[..., 0]  # raw confidence logit

        # build target tensors
        obj   = torch.zeros(N, S, S,     device=device)
        t_cx  = torch.zeros(N, S, S,     device=device)
        t_cy  = torch.zeros(N, S, S,     device=device)
        t_w   = torch.zeros(N, S, S,     device=device)
        t_h   = torch.zeros(N, S, S,     device=device)
        t_cls = torch.zeros(N, S, S, C,  device=device)

        for n in range(N):
            for box in targets[n]:
                c_, cx, cy, bw, bh = box.tolist()
                gi = min(int(cx * S), S - 1)
                gj = min(int(cy * S), S - 1)
                obj[n, gj, gi]            = 1.
                t_cx[n, gj, gi]           = cx
                t_cy[n, gj, gi]           = cy
                t_w[n, gj, gi]            = bw
                t_h[n, gj, gi]            = bh
                t_cls[n, gj, gi, int(c_)] = 1.

        obj_m = obj.bool()  # (N,S,S)

        # expand GT to (N,S,S,B) for IoU
        e = lambda t: t.unsqueeze(-1).expand(-1, -1, -1, B)
        tcx_e, tcy_e, tw_e, th_e = e(t_cx), e(t_cy), e(t_w), e(t_h)

        # IoU between each predicted box and GT (for responsible predictor selection)
        with torch.no_grad():
            px1 = pcx - pw/2;  px2 = pcx + pw/2
            py1 = pcy - ph/2;  py2 = pcy + ph/2
            gx1 = tcx_e - tw_e/2; gx2 = tcx_e + tw_e/2
            gy1 = tcy_e - th_e/2; gy2 = tcy_e + th_e/2
            iw  = (torch.min(px2, gx2) - torch.max(px1, gx1)).clamp(0)
            ih  = (torch.min(py2, gy2) - torch.max(py1, gy1)).clamp(0)
            inter = iw * ih
            iou = (inter / (pw*ph + tw_e*th_e - inter + 1e-8)).detach()  # (N,S,S,B)

        # responsible predictor = highest IoU per obj cell
        best     = iou.argmax(-1)                              # (N,S,S)
        resp     = torch.zeros(N, S, S, B, device=device)
        resp.scatter_(-1, best.unsqueeze(-1), 1.)              # one-hot (N,S,S,B)

        obj4     = obj_m.unsqueeze(-1).expand(-1,-1,-1,B)
        resp_obj = (resp * obj4.float()).bool()                 # responsible box in obj cells

        # coordinate loss (paper eq 1-2)
        # cell-relative predictions: sigmoid(raw) gives offset in [0,1] within cell
        pred_cx_rel = torch.sigmoid(boxes[..., 1])             # (N,S,S,B)
        pred_cy_rel = torch.sigmoid(boxes[..., 2])
        pred_w      = torch.sigmoid(boxes[..., 3])
        pred_h      = torch.sigmoid(boxes[..., 4])

        # targets in cell-relative coords
        tgt_cx_rel  = (t_cx * S - gx[None,...]).unsqueeze(-1).expand_as(pred_cx_rel)
        tgt_cy_rel  = (t_cy * S - gy[None,...]).unsqueeze(-1).expand_as(pred_cy_rel)
        tgt_w_e     = tw_e
        tgt_h_e     = th_e

        # sqrt trick for scale invariance (paper section 2, loss)
        coord_loss = (
            self.mse(pred_cx_rel[resp_obj], tgt_cx_rel[resp_obj])
          + self.mse(pred_cy_rel[resp_obj], tgt_cy_rel[resp_obj])
          + self.mse(
                pred_w[resp_obj].clamp(1e-8).sqrt(),
                tgt_w_e[resp_obj].clamp(1e-8).sqrt())
          + self.mse(
                pred_h[resp_obj].clamp(1e-8).sqrt(),
                tgt_h_e[resp_obj].clamp(1e-8).sqrt())
        )

        # object confidence loss (paper eq 3) — responsible box, target = IoU
        best_iou  = (iou  * resp).sum(-1)                      # (N,S,S)
        best_conf = (pc   * resp).sum(-1)                      # raw logit
        obj_conf_loss = self.mse(
            torch.sigmoid(best_conf[obj_m]),
            best_iou[obj_m])

        # no-object confidence loss (paper eq 4)
        # Applied to BOTH predictors in non-obj cells (matching the paper)
        noobj_m       = (~obj_m).unsqueeze(-1).expand_as(pc)   # (N,S,S,B)
        noobj_conf_loss = self.mse(
            torch.sigmoid(pc[noobj_m]),
            torch.zeros(noobj_m.sum(), device=device))

        # class loss (paper eq 5)
        cls_loss = self.mse(
            torch.sigmoid(cls_p[obj_m]),
            t_cls[obj_m])

        n_total = float(N * S * S * B)
        return (5 * coord_loss + obj_conf_loss + 0.5 * noobj_conf_loss + cls_loss) / n_total


criterion = YOLOv1Loss(S, B, C)


In [6]:
def decode(pred, conf_thresh=0.25, S=7, B=2, C=20):
    device = pred.device
    gy, gx = torch.meshgrid(
        torch.arange(S, device=device, dtype=torch.float32),
        torch.arange(S, device=device, dtype=torch.float32),
        indexing='ij')

    boxes = pred[0, ..., :B*5].reshape(S, S, B, 5)
    cls_p = pred[0, ..., B*5:]

    cx   = (torch.sigmoid(boxes[..., 1]) + gx[:, :, None]) / S
    cy   = (torch.sigmoid(boxes[..., 2]) + gy[:, :, None]) / S
    w    = torch.sigmoid(boxes[..., 3])
    h    = torch.sigmoid(boxes[..., 4])
    conf = torch.sigmoid(boxes[..., 0])

    cls_prob = torch.softmax(cls_p, dim=-1)
    scores   = conf.unsqueeze(-1) * cls_prob.unsqueeze(2)   # (S,S,B,C)
    max_sc, cls_id = scores.max(-1)                          # (S,S,B)
    mask = max_sc > conf_thresh

    cx_f = cx[mask]; cy_f = cy[mask]; w_f = w[mask]; h_f = h[mask]
    sc_f = max_sc[mask]; cl_f = cls_id[mask]

    boxes_xyxy = torch.stack([cx_f-w_f/2, cy_f-h_f/2, cx_f+w_f/2, cy_f+h_f/2], dim=-1)
    return boxes_xyxy, sc_f, cl_f


def nms(boxes, scores, iou_thresh=0.45):
    if boxes.numel() == 0:
        return torch.zeros(0, dtype=torch.long)
    x1,y1,x2,y2 = boxes[:,0],boxes[:,1],boxes[:,2],boxes[:,3]
    areas = (x2-x1)*(y2-y1)
    order = scores.argsort(descending=True)
    keep  = []
    while order.numel():
        i = order[0].item(); keep.append(i)
        if order.numel() == 1: break
        rest = order[1:]
        iw = (torch.min(x2[i], x2[rest]) - torch.max(x1[i], x1[rest])).clamp(0)
        ih = (torch.min(y2[i], y2[rest]) - torch.max(y1[i], y1[rest])).clamp(0)
        iou = iw*ih / (areas[i] + areas[rest] - iw*ih + 1e-8)
        order = rest[iou <= iou_thresh]
    return torch.tensor(keep)


def compute_map(val_dl, conf_thresh=0.25, iou_thresh=0.45, map_iou=0.5):
    model.eval()
    all_pred, all_gt = [], []
    img_id = 0
    with torch.no_grad():
        for imgs, targets in tqdm(val_dl, desc='  mAP', leave=False):
            imgs  = imgs.to(DEVICE)
            preds = model(imgs)
            for i in range(imgs.shape[0]):
                pb, ps, pc = decode(preds[i:i+1], conf_thresh, S, B, C)
                pb = pb.cpu(); ps = ps.cpu(); pc = pc.cpu()
                if pb.numel() > 0:
                    keep = nms(pb, ps, iou_thresh)
                    for k in keep:
                        all_pred.append([img_id, pc[k].item(), ps[k].item()] + pb[k].tolist())
                for box in targets[i]:
                    c_, cx, cy, bw, bh = box.tolist()
                    # store as [img_id, cls, x1, y1, x2, y2]
                    all_gt.append([img_id, int(c_), cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2])
                img_id += 1

    if not all_pred or not all_gt:
        return 0.0

    aps = []
    for cls in range(C):
        gt_c = {r[0]: [] for r in all_gt if r[1] == cls}
        pd_c = sorted([r for r in all_pred if r[1] == cls], key=lambda x: -x[2])
        if not gt_c or not pd_c:
            continue
        for iid in gt_c:
            gt_c[iid] = [r[2:] for r in all_gt if r[0] == iid and r[1] == cls]  # [x1,y1,x2,y2]
        matched = {iid: [False]*len(v) for iid, v in gt_c.items()}
        tp = torch.zeros(len(pd_c)); fp = torch.zeros(len(pd_c))
        for j, det in enumerate(pd_c):
            iid = det[0]; db = det[3:]   # [x1,y1,x2,y2]
            if iid not in gt_c or not gt_c[iid]:
                fp[j] = 1; continue
            best_iou, best_k = 0.0, -1
            for k, gb in enumerate(gt_c[iid]):
                iw = max(0, min(db[2],gb[2]) - max(db[0],gb[0]))
                ih = max(0, min(db[3],gb[3]) - max(db[1],gb[1]))
                ii = iw * ih
                u  = (db[2]-db[0])*(db[3]-db[1]) + (gb[2]-gb[0])*(gb[3]-gb[1]) - ii
                iou_v = ii / (u + 1e-8)
                if iou_v > best_iou:
                    best_iou, best_k = iou_v, k
            if best_iou >= map_iou and best_k >= 0 and not matched[iid][best_k]:
                tp[j] = 1; matched[iid][best_k] = True
            else:
                fp[j] = 1
        n_gt   = sum(len(v) for v in gt_c.values())
        cum_tp = tp.cumsum(0); cum_fp = fp.cumsum(0)
        prec   = cum_tp / (cum_tp + cum_fp + 1e-8)
        rec    = cum_tp / (n_gt + 1e-8)
        ap = 0.
        for t in torch.linspace(0, 1, 11):
            p_at_r = prec[rec >= t]
            ap += (p_at_r.max().item() if p_at_r.numel() else 0.) / 11
        aps.append(ap)

    return float(np.mean(aps)) if aps else 0.0


In [7]:
def train_run(train_dl, val_dl, epochs, freeze_epochs=10):
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[int(epochs*0.6), int(epochs*0.8)], gamma=0.1)
    best_map = 0.0
    hist = {'train': [], 'val': [], 'map': []}

    for epoch in range(1, epochs + 1):
        if freeze_epochs > 0 and epoch == freeze_epochs + 1:
            freeze_backbone(model, freeze=False)

        model.train()
        t_loss = 0.0
        bar = tqdm(train_dl, desc=f'  {epoch}/{epochs} train', leave=False)
        for imgs, targets in bar:
            imgs = imgs.to(DEVICE)
            loss = criterion(model(imgs), targets)
            opt.zero_grad(); loss.backward(); opt.step()
            t_loss += loss.item()
            bar.set_postfix(loss=f'{loss.item():.3f}')
        t_loss /= len(train_dl)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for imgs, targets in tqdm(val_dl, desc=f'  {epoch}/{epochs} val', leave=False):
                v_loss += criterion(model(imgs.to(DEVICE)), targets).item()
        v_loss /= len(val_dl)

        curr_map = compute_map(val_dl)
        saved = curr_map > best_map
        if saved:
            best_map = curr_map
            torch.save(model.state_dict(), 'yolov1_best.pt')

        hist['train'].append(t_loss)
        hist['val'].append(v_loss)
        hist['map'].append(curr_map)

        sched.step()
        print(f'epoch {epoch:>3}/{epochs}  train={t_loss:.3f}  val={v_loss:.3f}  mAP50={curr_map:.4f}  best={best_map:.4f}{"  *" if saved else ""}', flush=True)

    return hist


In [8]:
model = YOLOv1(S, B, C).to(DEVICE)
freeze_backbone(model, freeze=True)
hist = train_run(train_dl, val_dl, EPOCHS, freeze_epochs=10)


backbone frozen


epoch   1/30  train=0.063  val=0.049  mAP50=0.1626  best=0.1626  *


epoch   2/30  train=0.047  val=0.045  mAP50=0.2540  best=0.2540  *


epoch   3/30  train=0.043  val=0.042  mAP50=0.2253  best=0.2540


epoch   4/30  train=0.040  val=0.041  mAP50=0.2266  best=0.2540


...

epoch  26/30  train=0.022  val=0.031  mAP50=0.2932  best=0.2932  *


epoch  27/30  train=0.021  val=0.031  mAP50=0.2972  best=0.2972  *


epoch  28/30  train=0.021  val=0.031  mAP50=0.2928  best=0.2972


epoch  29/30  train=0.021  val=0.031  mAP50=0.2931  best=0.2972


epoch  30/30  train=0.021  val=0.031  mAP50=0.2890  best=0.2972
